In [0]:
%run ./10-kafka-to-bronze

In [0]:
class kafkaToBronzeTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.invoices_bz")
        dbutils.fs.rm(f"{self.checkpoint_dir}/invoices_bz",True)
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql("select count(*) from spark_stream.default.invoices_bz").collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done") 

    def runTests(self):
        self.cleanTests()
        bzStream = Bronze()        

        print("Testing Scenario - Start from beginning on a new checkpoint...") 
        bzQuery = bzStream.process()
        bzQuery.awaitTermination()    
        self.assertResult(30)
        print("Validation passed.\n")        

        print("Testing Scenario - Restart from where it stopped on the same checkpoint...")
        bzQuery = bzStream.process()
        bzQuery.awaitTermination()    
        self.assertResult(30)
        print("Validation passed.\n")     

        print("Testing Scenario - Start from 1794656434964 on a new checkpoint...") 
        self.cleanTests()
        bzQuery = bzStream.process(1794656434964)
        bzQuery.awaitTermination()    
        self.assertResult(20)
        print("Validation passed.\n")    

In [0]:
ts = kafkaToBronzeTestSuite()
ts.runTests()